The contents of this notebook were created with assistance from Claude generative AI.

# 🏷️ Building a Golden Test Set to Referee Two LLM Labelers
### NYC congestion-pricing stance study — SIADS 696 Milestone 2

This notebook explains, end-to-end, **how we built a small human-labeled "gold" test set** and used it to
decide which of two automatic labelers to trust. It runs on a 12-row toy dataset so you can follow the
logic without the real ~2 GB of Reddit data. The real pipeline lives in ``.


## The situation

We labeled **~1.3 M Reddit comments/posts** for *stance* toward NYC congestion pricing (the ~$9 toll into
lower Manhattan). Labels came from two models:

| labeler | what | cost | trust |
|---|---|---|---|
| **7B / vLLM** | a small local model, ran on everything | cheap, fast | weak |
| **Claude Sonnet** | re-labeled a 5,131-item sample | ~$21 batch | stronger |

The catch: **they disagree a lot** — stance Cohen's κ ≈ **0.22**, they differ on stance for ~59% of items
and on relevance for ~26%. We can't just pick one. So we build a **human gold set** as the tiebreaker —
and we sample it cleverly so a few hundred human labels go as far as possible.


## Step 1 — What do we sample *from*?

Our first instinct was to sample from whole "relevant threads." That was a trap: clicking through such a
queue, **~65% of items were off-topic.** A single comment mentioning the toll makes the *whole thread* a
"hit," and the reply chain wanders off into Daniel Penny, subway germs, German trams…

The fix: only sample **item-level lexical anchors** — items whose *own text* names congestion pricing
(`hp` comments) or posts that matched the topic filter (`matched_tier1`). A spot check found these are
**~100% on-topic.** That's the population both models labeled in the 5,131-item set below.


In [1]:
import pandas as pd
from IPython.display import HTML, display

# A tiny stand-in for the 5,131 item-level items. Each carries BOTH models' labels.
# topic = about_topic (1 = engages with the toll). Real set: 4,631 comments + 500 posts.
items = pd.DataFrame([
  # id,  kind,     context (post + ancestor chain),                         target,                                                  7B_topic,7B_stance, C_topic,C_stance
  ('c01','comment','[POST·OP] Congestion pricing starts Jan 5\n[A] Finally, some sanity.','[B] $9 a day just to get to work is robbery.',            1,'anti',   1,'anti'),
  ('c02','comment','[POST·OP] Two weeks in — how is traffic?\n[A] Crosstown buses move now.','[B] Best thing to happen to this city in a decade.',      1,'pro',    1,'pro'),
  ('c03','comment','[POST·OP] Toll cameras are live\n[A] Total cash grab.','[B] This. Exactly this.',                                                  1,'anti',   1,'anti'),
  ('c04','comment','[POST·OP] Congestion pricing is good for the US\n[A] No it is not lol.','[B] Say it louder for the anti-car people in the back.',     1,'pro',    1,'anti'),  # D_flip
  ('c05','comment','[POST·OP] Hochul revives congestion pricing','[B] Good policy, but $9 is way too low to change behavior.',                           1,'mixed',  1,'pro'),   # D_stance
  ('c06','comment','[POST·OP] Congestion pricing FAQ','[B] Does the toll apply on weekends or only weekday peak?',                                       1,'pro',    1,'neutral'),# D_stance
  ('c07','comment','[POST·OP] Congestion pricing megathread\n[A] About time.','[B] Btw the BQE speed cameras ticketed me 3x this month.',               1,'neutral',0,'NA'),    # D_topic (7B wrong)
  ('c08','comment','[POST·OP] More CP anecdotes\n[A] Trips are faster now.','[B] A lot of toll revenue funds rail — what if it drops?',                  0,'NA',     1,'neutral'),# D_topic (7B dropped a real hit)
  ('c09','comment','[POST·OP] Toll zone map released\n[A] My commute will get pricey.','[B] We need residential parking permits like other cities.',     1,'anti',   0,'NA'),    # D_topic
  ('c10','comment','[POST·OP] Congestion pricing thread\n[A] Finally.','[B] Unrelated — why is the R train so unreliable lately?',                       1,'pro',    0,'NA'),    # D_topic
  ('p01','post',   '(top-level post — no parent context)','[OP] This congestion tax is class warfare against working people.',                          1,'anti',   1,'anti'),
  ('p02','post',   '(top-level post — no parent context)','[OP] MTA reports congestion-zone traffic down 7.5% in month one.',                            1,'neutral',1,'neutral'),
], columns=['id','kind','context','target','vllm_topic','vllm_stance','claude_topic','claude_stance'])
items[['id','kind','target','vllm_stance','claude_stance']]


,id,kind,target,vllm_stance,claude_stance
0,c01,comment,[B] $9 a day just to get to work is robbery.,anti,anti
1,c02,comment,[B] Best thing to happen to this city in a dec...,pro,pro
2,c03,comment,[B] This. Exactly this.,anti,anti
3,c04,comment,[B] Say it louder for the anti-car people in t...,pro,anti
4,c05,comment,"[B] Good policy, but $9 is way too low to chan...",mixed,pro
5,c06,comment,[B] Does the toll apply on weekends or only we...,pro,neutral
6,c07,comment,[B] Btw the BQE speed cameras ticketed me 3x t...,neutral,NA
7,c08,comment,[B] A lot of toll revenue funds rail — what if...,NA,neutral
8,c09,comment,[B] We need residential parking permits like o...,anti,NA
9,c10,comment,[B] Unrelated — why is the R train so unreliab...,pro,NA


## Step 2 — Find *where the models disagree*

A human label is only decisive where the two models **conflict**. So we partition every item into four cells:

| cell | meaning | in the real 5,131 |
|---|---|---|
| `A_agree`  | both agree (stance **and** topic) | 1,999 |
| `D_flip`   | both on-topic, **pro ↔ anti** flip (the hardest) | 597 |
| `D_stance` | both on-topic, stance differs (e.g. pro↔neutral) | 1,185 |
| `D_topic`  | disagree on *relevance* (95% are 7B-off / Sonnet-on) | 1,350 |


In [2]:
def cell_of(r):
    if bool(r.vllm_topic) != bool(r.claude_topic):      # disagree on relevance
        return 'D_topic'
    if r.vllm_stance == r.claude_stance:                # agree on stance + topic
        return 'A_agree'
    if {r.vllm_stance, r.claude_stance} == {'pro', 'anti'}:
        return 'D_flip'                                 # the hardest disagreement
    return 'D_stance'

items['cell'] = items.apply(cell_of, axis=1)
items['cell'].value_counts()


cell
A_agree     5
D_topic     4
D_stance    2
D_flip      1
Name: count, dtype: int64

## Step 3 — Sample, **oversampling the disagreements**

We build the queue in **rounds of 50** with a fixed cell mix, so *any* whole-round prefix is still a valid
sample (the annotator can stop early). Disagreement cells are oversampled — `D_flip` at ~28% vs `A_agree`
at ~8% — because that's where the human's answer actually changes the outcome. We record each item's cell
so we can **re-weight** back to unbiased estimates later (`w = N_cell / n_labeled_in_cell`).
Round 1 is labeled by **all three raters** for an inter-annotator agreement check.


In [3]:
import random
# toy per-round quota (the real one is 12/13/13/12 over 14 rounds = 700 items)
QUOTA = {'D_flip': 1, 'D_stance': 1, 'D_topic': 2, 'A_agree': 1}
queue = pd.concat([items[items.cell == c].sample(frac=1, random_state=0).head(q)
                   for c, q in QUOTA.items()])
queue = queue.sample(frac=1, random_state=1).reset_index(drop=True)        # shuffle within the round
queue['qid'] = [f'q{i+1:03d}' for i in range(len(queue))]
queue[['qid', 'cell', 'kind', 'target']]


,qid,cell,kind,target
0,q001,D_topic,comment,[B] We need residential parking permits like o...
1,q002,D_stance,comment,[B] Does the toll apply on weekends or only we...
2,q003,A_agree,comment,[B] This. Exactly this.
3,q004,D_flip,comment,[B] Say it louder for the anti-car people in t...
4,q005,D_topic,comment,[B] Unrelated — why is the R train so unreliab...


## Step 4 — The labeling UI (deliberately **blind**)

The annotator opens a single self-contained `adjudication_tool.html` and sees **only the context + the
highlighted target** — never the model labels (those stay in a local sidecar, so the human can't be
anchored). It saves every keystroke to a file (resumable), supports keyboard `1`–`6`, and shows the exact
text the models saw. Here's a one-item mock of what the annotator sees:


In [4]:
def render_card(row):
    box = 'background:#fff;border-radius:8px;padding:10px 12px;white-space:pre-wrap;font-size:14px'
    ctx = '' if row.kind == 'post' else (
        "<div style='color:#666;font-size:12px;margin:6px 0 2px'>CONTEXT — post &amp; reply chain</div>"
        "<div style='" + box + ";border-left:4px solid #2563eb'>" + row.context + "</div>")
    tgt = ("<div style='color:#666;font-size:12px;margin:10px 0 2px'>&#9660; LABEL THE HIGHLIGHTED ITEM</div>"
           "<div style='" + box + ";background:#fff7e6;border:2px solid #f59e0b'>" + row.target + "</div>")
    btns = ''.join("<span style='border:1px solid #ddd;border-radius:8px;padding:4px 9px;margin-right:5px;"
                   "font-size:12px'>" + f'{i+1} {n}</span>'
                   for i, n in enumerate(['PRO','ANTI','NEUTRAL','MIXED','NOT ABOUT IT','UNSURE']))
    head = "<div style='color:#888;font-size:12px;margin-bottom:4px'>" + row.kind.upper() + \
           " &middot; the annotator never sees the 7B / Sonnet labels</div>"
    return HTML("<div style='font-family:system-ui;max-width:680px'>" + head + ctx + tgt +
                "<div style='margin-top:10px'>" + btns + "</div></div>")

display(render_card(queue.iloc[0]))


### The codebook (shared by the humans *and* both LLMs)

Everyone labels against one definition, so the labels are comparable:

- **PRO** — supports the toll existing (even while griping about the price). *"Best thing for midtown."*
- **ANTI** — opposes the toll. *"A cash grab on working people."*
- **NEUTRAL** — on-topic, no side: a question or news fact. *"Does it apply on weekends?"*
- **MIXED** — genuinely argues both sides. *"Great for traffic, but the exemptions are unfair."*
- **NOT ABOUT IT (NA)** — not about the toll, even inside a relevant thread (speed cameras, parking permits).

Key rules: use the reply chain ("me too" inherits the parent's stance); stance is toward *the toll*, not
toward Hochul/the MTA/drivers; "good but $9 is too low" is **pro**, not mixed.


## Step 5 — Score: **who does the human back?**

Once labels come back (a JSONL export from the tool), we score *both* models against the human truth.
The headline is the **head-to-head per cell**: on the items where the models disagreed, what fraction does
the human's label side with each one? (Below we hand-set a plausible "human truth"; in reality it's the
export. `A_agree` is a check that the easy region is actually correct.)


In [5]:
# Pretend a human labeled the full set (in practice: the tool's JSONL export).
truth = {'c01':'anti','c02':'pro','c03':'anti','c04':'anti','c05':'pro','c06':'neutral',
         'c07':'NA','c08':'neutral','c09':'NA','c10':'NA','p01':'anti','p02':'neutral'}
items['human'] = items.id.map(truth)

def who_is_right(r):
    if r.cell == 'D_topic':                    # relevance disagreement -> compare about_topic
        h = r.human != 'NA'
        return pd.Series({'7B_right': bool(r.vllm_topic) == h, 'Sonnet_right': bool(r.claude_topic) == h})
    return pd.Series({'7B_right': r.human == r.vllm_stance, 'Sonnet_right': r.human == r.claude_stance})

verdict = items.join(items.apply(who_is_right, axis=1))
print('Head-to-head — fraction the human backs each model, by cell:')
verdict.groupby('cell')[['7B_right', 'Sonnet_right']].mean().round(2)


Head-to-head — fraction the human backs each model, by cell:


,7B_right,Sonnet_right
cell,,
A_agree,1.0,1.0
D_flip,0.0,1.0
D_stance,0.0,1.0
D_topic,0.0,1.0


In this toy run the human sides with **Sonnet** on essentially every disagreement — which is exactly the
pattern we expect to measure for real (the local 7B drops on-topic items and flips stances). The real
`score_relabel.py` also weights each cell back to the full 5,131 and reports per-class precision/recall/F1.


### Inter-annotator agreement (are the *humans* even consistent?)

Three people label round 1; we report Cohen's κ pairwise (and Fleiss' κ across all three). If the humans
don't agree, the codebook needs work before we trust any verdict.


In [6]:
def cohen_kappa(a, b):
    a, b = pd.Series(a), pd.Series(b)
    po = (a.values == b.values).mean()
    pe = sum((a == c).mean() * (b == c).mean() for c in set(a) | set(b))
    return (po - pe) / (1 - pe)

rater_A = ['anti', 'pro', 'anti', 'anti', 'pro', 'neutral']
rater_B = ['anti', 'pro', 'anti', 'neutral', 'pro', 'neutral']   # disagree on one
print('raw agreement:', round((pd.Series(rater_A) == pd.Series(rater_B)).mean(), 2))
print("Cohen's kappa:", round(cohen_kappa(rater_A, rater_B), 3))


raw agreement: 0.83
Cohen's kappa: 0.75


## How the toy maps to the real pipeline

| this notebook | real file (``) |
|---|---|
| `items` (12 rows, both labels) | `relabel_5k-claude.csv` (5,131 rows: 7B + Sonnet) |
| `cell_of` + round sampling | `build_gold_from_5k.py` |
| `render_card` mock | `adjudication_tool.html` (real, blind, resumable) |
| `who_is_right` + `cohen_kappa` | `score_relabel.py` |

**Caveats the real pipeline handles:** Horvitz-Thompson weights so the oversampled disagreements don't
bias population estimates; the model labels are kept entirely out of the HTML (blind); and the human is
shown *fuller* text than Sonnet saw (the gold set should have the best context for a correct call).

**Bottom line:** a few hundred human labels, concentrated on the contested items, are enough to declare
which automatic label set we trust as the training data — without hand-labeling all 5,131.
